Language composition

This report shows the different languages that are used in the projects.

In [ ]:
unit: str = "files"

In [ ]:
import plotly.express as px
from moderne_visualizations_misc.reusable.language_composition import (
    read_language_composition,
)
import code_data_science.palette as palette

# One row per group of files, read in chunks so that large tables fit in memory
df = read_language_composition(
    "../samples/v2/org.openrewrite.table.LanguageCompositionPerFile.csv",
    by_repository=False,
)

# Exit early if there are no rows and render a plot with a message
if len(df) == 0:
    fig = px.treemap(names=[], parents=[])
    fig.add_annotation(
        x=0.5, y=0.5, text="No rows of data found", showarrow=False, font=dict(size=20)
    )
else:
    df_file_grouped = df.groupby(
        by=["language", "fileType", "sourceFileType", "hasParseFailures"],
        as_index=False,
    ).sum(numeric_only=True)

    path = ["language", "fileType"]

    is_unparseable = df["language"] == "Other/unknown/unparseable"
    is_java = df["fileType"] == "java"
    has_parse_failures = df["hasParseFailures"] == True  # noqa: E712

    # number of files in the data table
    total_files = df["files"].sum()

    # number of files with a language of Other/unknown/unparseable
    total_unparseable_files = df.loc[is_unparseable, "files"].sum()

    # number of files with a file type of java but a language of Other/unknown/unparseable
    total_java_unparseable_files = df.loc[is_unparseable & is_java, "files"].sum()

    # number of files with parse failures
    total_parse_failures = df.loc[has_parse_failures, "files"].sum()

    # number of files with a file type of java and parse failures
    total_java_parse_failures = df.loc[has_parse_failures & is_java, "files"].sum()

    # determine measurement field based on `unit`
    measurement_field = "files" if unit == "files" else "linesOfText"

    # sum of unit in java files affected by parse failures
    total_lines_of_java_affected_by_parse_failures = df.loc[
        has_parse_failures & is_java, measurement_field
    ].sum()

    # Set the color palette
    colors = palette.qualitative()

    fig = px.treemap(
        df_file_grouped,
        path=path,
        values=measurement_field,
        color_discrete_sequence=colors,
    )

    fig.update_layout(
        margin=dict(t=28, l=0, r=0, b=0),
        annotations=[
            dict(
                x=0.01,
                y=1.055,
                showarrow=False,
                borderwidth=1,
                text="<b>More info</b>",
                hovertext=(
                    f"""
            <b>Total files:</b> {"{:,}".format(total_files)} </br> </br>
            <b>Other/unknown/unparseable files:</b> {"{:,}".format(total_unparseable_files)}</br>
            <b>Other/unknown/unparseable .java files:</b> {"{:,}".format(total_java_unparseable_files)} </br>
            <b>Files with parse failures:</b> {"{:,}".format(total_parse_failures)} </br>
            <b>.java files with parse failures:</b> {"{:,}".format(total_java_parse_failures)} </br>
            <b>Total lines of text in .java files with parse failures:</b> {"{:,}".format(total_lines_of_java_affected_by_parse_failures)}
            """
                ),
                xref="paper",
                yref="paper",
            )
        ],
    )

    fig.update_traces(
        marker=dict(cornerradius=3),
    )

    fig.data[0].textinfo = "label+text+value"

In [ ]:
# Output the visualization
fig.show(
    render="plotly_mimetype",
    config={
        "toImageButtonOptions": {
            "filename": "LanguageCompositionPerFile-LanguageComposition"
        }
    },
)